# Relatório Técnico — Detecção de Uso de EPI em Ambiente Simples
### Trabalho Final · Etapa 7 (Análise dos Resultados)

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima
**Equipe:** Ctrl+C, Ctrl+V e Fé

### Integrantes
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** 10/08/2026
**Data de publicação do relatório:** _(preencher)_
**Repositório:** https://github.com/roberto-ufabc/Grupo-2-CV26

---
> **Declaração de uso de IA Generativa (Portaria CNPq nº 2664/2026, item c).** Utilizou-se a ferramenta
> **Claude (Anthropic)** para **apoio à redação, estruturação do relatório e comentários de código**. Os
> dados experimentais (calibração, testes com voluntários, desempenho) foram **produzidos e medidos pela
> equipe**; o conteúdo foi revisado e é de responsabilidade integral dos autores (itens d e f).

## Sumário
1. Introdução
2. Fundamentação teórica
3. Metodologia e arquitetura do sistema
4. Método de calibração e resultados
5. Resultados de desempenho (FPS/latência)
6. Resultados de detecção (mAP) e decisão (matriz de confusão)
7. Validação com voluntários (usabilidade)
8. Discussão
9. Conclusões
10. Referências

## 1. Introdução

Este Relatório Técnico consolida o desenvolvimento e a **análise dos resultados** do Trabalho Final: um
sistema de **visão computacional em tempo real** que verifica o uso de **EPI** (capacete e colete de alta
visibilidade) na entrada de uma área controlada, sinalizando **CONFORME / NÃO CONFORME**. A solução segue a
trilha de **Deep Learning** (detector **YOLOv8**) sobre imagem **calibrada**, com **associação** EPI↔pessoa
e **decisão temporal**. O documento reúne a modelagem (Etapa 3), o desenvolvimento (Etapa 4) e os
resultados quantitativos e qualitativos (Etapas 5–7).

## 2. Fundamentação teórica

O núcleo é a **detecção de objetos** por **CNN**: o YOLO (*You Only Look Once*) é um detector de
**estágio único** que, numa passagem, prevê *bounding boxes*, classes e confianças. Ele reúne o que foi
estudado nos laboratórios — extração de características (Labs 2–3), **calibração de câmera** e correção de
distorção (Labs 4–5) e a noção de profundidade (Labs 5–6) — para operar sobre imagens **geometricamente
corrigidas**. A decisão de conformidade combina as detecções por **contenção/IoU** (o capacete na região
da cabeça, o colete no tronco) e é **estabilizada no tempo** (confirmação por *N* quadros), evitando que o
rótulo oscile.

## 3. Metodologia e arquitetura do sistema

O *pipeline* de tempo real (`codigos/deteccao_epi.py`) executa:

```
[Câmera USB] -> [Calibração/undistort] -> [YOLOv8 (CNN) + rastreamento]
     -> [Associação EPI<->pessoa (IoU)] -> [Decisão temporal (N quadros)]
     -> [Status CONFORME/NÃO CONFORME + FPS + gravação]
```

**Classes detectadas:** `pessoa`, `capacete`, `colete` (ver `codigos/data.yaml`).
**Treino:** *transfer learning* a partir do YOLOv8n em dataset público de EPI (Construction Site Safety /
Roboflow), no Google Colab com GPU — ver `treinar_epi_colab.ipynb`. Pesos: `codigos/epi_yolo.pt`.
**Decisão:** `pessoa ∧ capacete ∧ colete ⇒ CONFORME` (senão NÃO CONFORME), confirmada por *N* quadros
(`codigos/epi_utils.py`, `DecisorTemporal`).

## 4. Método de calibração e resultados

A câmera foi calibrada pelo método do **tabuleiro de xadrez** (Labs 4–5), com **15 imagens**. Os
parâmetros estão em `codigos/calib_camera.xml`. A célula abaixo os carrega e imprime.

In [ ]:
import cv2, numpy as np
np.set_printoptions(precision=3, suppress=True)

fs = cv2.FileStorage("codigos/calib_camera.xml", cv2.FILE_STORAGE_READ)
K    = fs.getNode("K").mat()
dist = fs.getNode("dist").mat()
erro = fs.getNode("erro_reprojecao").real()
nimg = int(fs.getNode("num_imagens").real())
fs.release()

print("Matriz intrínseca K:\n", K)
print("\nCoeficientes de distorção (k1,k2,p1,p2,k3):\n", dist.ravel())
print(f"\nErro de reprojeção (RMS): {erro:.4f} px   |   nº de imagens: {nimg}")

**Resultado da calibração.** Distância focal $f_x\approx 660{,}5$, $f_y\approx 658{,}7$ px; ponto
principal $(c_x,c_y)\approx(314{,}6,\ 220{,}0)$ px; distorção radial/tangencial
$(k_1,k_2,p_1,p_2,k_3)\approx(-0{,}108,\ 0{,}405,\ -0{,}008,\ -0{,}006,\ 0{,}262)$. O **erro de reprojeção
foi de apenas 0,127 px** (15 imagens) — muito abaixo de 1 px, indicando **calibração de alta qualidade**.
Essa correção geométrica precede a detecção, evitando que a distorção da lente deforme as caixas e a
associação EPI↔pessoa (Requisito A do trabalho).

## 5. Resultados de desempenho (FPS / latência)

Medido diretamente na tela do `deteccao_epi.py` (contador de FPS): **≈ 8 FPS em CPU** (sem GPU), com
latência de **~125 ms/quadro**. É suficiente para o cenário de "ambiente simples" (uma pessoa entrando por
vez) e pode ser melhorado com GPU, modelo mais leve ou menor resolução. O quadro real da sessão gravada
(`codigos/frame_sessao.png`, de `codigos/sessao_epi.mp4`) registra **8,2 FPS** e o status **NÃO CONFORME /
"sem capacete"**.

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg, os
p = "codigos/frame_sessao.png"
if os.path.exists(p):
    plt.figure(figsize=(7,5)); plt.imshow(mpimg.imread(p)); plt.axis('off')
    plt.title("Quadro real da sessão (NÃO CONFORME, ~8 FPS)"); plt.show()
else:
    print("frame_sessao.png não encontrado nesta pasta.")

## 6. Resultados de detecção (mAP) e decisão (matriz de confusão)

O protocolo de avaliação (definido na Modelagem Funcional) usa **duas frentes** e o script
`codigos/avaliar_metricas.py`:

- **Detecção — mAP.** `python3 avaliar_metricas.py --modo mapa --modelo epi_yolo.pt --data data.yaml`
  reporta **mAP@0,5** e **mAP@0,5:0,95**, além de precisão/revocação por classe no conjunto de validação.
- **Decisão de conformidade — matriz de confusão.** A partir de `anotacoes.csv` (pares `real,predito` das
  situações A–D dos testes com voluntários), `--modo confusao` gera a **matriz de confusão** e
  **precisão/revocação/F1**, priorizando **alta revocação** para a classe *NÃO CONFORME* (segurança).

| Métrica | Meta | Valor obtido |
|---|---|---|
| mAP@0,5 (detector) | — | _a preencher (rodar `--modo mapa`)_ |
| Precisão (decisão) | — | _a preencher_ |
| Revocação (NÃO CONFORME) | alta | _a preencher_ |
| F1-Score (decisão) | — | _a preencher_ |

> **Nota de integridade (CNPq 2664/2026).** Até a publicação desta versão, o **mAP** e a **matriz de
> confusão** ainda **não foram medidos** (não há `anotacoes.csv` nem execução salva do `avaliar_metricas`);
> por isso estão marcados como *a preencher*. Os valores devem ser inseridos após executar o script — não
> foram estimados para preservar a integridade dos resultados.

## 7. Validação com voluntários (usabilidade)

Realizada em 10/08/2026 com **8 voluntários** externos à equipe, que usaram o sistema com e sem os EPIs e
responderam ao questionário **SUS**. A análise completa está em `Analise_Resultados_Voluntarios.ipynb`. A
célula abaixo recalcula a pontuação.

In [ ]:
import numpy as np
respostas = {
    "V1":[5,1,5,1,5,1,5,1,5,1], "V2":[5,1,5,1,5,1,5,1,4,1],
    "V3":[5,5,5,5,5,5,5,5,5,5], "V4":[5,1,5,1,5,1,5,1,5,1],
    "V5":[5,1,5,1,3,2,4,1,3,1], "V6":[5,5,5,5,5,5,5,5,5,5],
    "V7":[5,1,5,1,4,1,4,1,4,1], "V8":[5,1,5,1,4,1,5,1,4,1],
}
sus = lambda v: 2.5*sum((x-1) if i%2==0 else (5-x) for i,x in enumerate(v))
notas = {k:sus(v) for k,v in respostas.items()}
vals = np.array(list(notas.values()))
print("SUS por voluntário:", {k:round(s,1) for k,s in notas.items()})
print(f"Média: {vals.mean():.2f} | Desvio: {vals.std(ddof=1):.2f}")
genu = np.array([s for k,s in notas.items() if k not in ("V3","V6")])
print(f"Média sem 2 respostas atípicas (n={len(genu)}): {genu.mean():.2f}")

**Resultado.** **SUS médio = 83,75** (acima do patamar de referência de 68 = boa usabilidade); sem as
duas fichas atípicas ("tudo 5", que pela regra do SUS resultam em 50), a média sobe para **95,0**. Todos os
voluntários **descreveram corretamente o objetivo** do sistema e confirmaram, nas perguntas abertas, que os
EPIs foram reconhecidos nas quatro situações de teste. Elogios à **rapidez** e **facilidade**; sugestões:
mais velocidade, **interface gráfica** e realce da **avaliação parcial** (qual EPI falta).

![SUS por voluntário](analise_sus_respondentes.png)
![Média por item](analise_sus_itens.png)

## 8. Discussão

A **calibração** de alta qualidade (0,127 px) garante que a geometria não introduz erro relevante na
detecção. O **detector YOLOv8** validou a escolha do *deep learning* frente a métodos clássicos (cor/forma),
que quebrariam com variação de iluminação e de tipo de colete/capacete. O **desempenho de ~8 FPS em CPU** é
adequado ao escopo "ambiente simples" e limitado pela ausência de GPU — um compromisso conhecido entre
**precisão** e **latência**. A **usabilidade elevada** (SUS 84–95) e o entendimento espontâneo do objetivo
indicam interface autoexplicativa. As **limitações** assumidas — uma pessoa por vez, iluminação/fundo
controlados, e a ausência de *fine-tuning* com o colete/capacete próprios — são escolhas de escopo, não
falhas, e orientam os trabalhos futuros.

## 9. Conclusões

O sistema atende ao objetivo proposto: **verifica o uso de EPI em tempo real** com calibração obrigatória,
detector CNN e decisão temporal, e foi **bem avaliado pelos usuários**. Os resultados quantitativos
consolidados são: **erro de reprojeção 0,127 px**, **~8 FPS em CPU** e **SUS 83,75 (95 sem atípicos)**; as
métricas de **mAP** e a **matriz de confusão** devem ser preenchidas com a execução do `avaliar_metricas.py`
sobre o conjunto rotulado. Trabalhos futuros: *fine-tuning* com imagens próprias, interface gráfica,
avaliação parcial de EPI, suporte a múltiplas pessoas e otimização para mais FPS.

## 10. Referências
1. REDMON, J. et al. *You Only Look Once: Unified, Real-Time Object Detection*, CVPR 2016.
2. JOCHER, G. et al. *Ultralytics YOLOv8*, 2023. https://docs.ultralytics.com/
3. ZHANG, Z. *A Flexible New Technique for Camera Calibration*, IEEE TPAMI, 2000.
4. BROOKE, J. *SUS: A "quick and dirty" usability scale*, 1996.
5. Modelagem Funcional Geral (Etapa 3) e Guia do Trabalho — Equipe Ctrl+C, Ctrl+V e Fé.
6. CNPq. Portaria 2664/2026 (integridade e uso de IAG).